In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

In [9]:
df = pd.read_csv("datasets/final_clustering.csv")

In [10]:
df_2 = pd.read_csv("datasets/df_sberne.csv")

In [11]:
df.head()

,region_name,mo,23-01,23-02,23-03,23-04,23-05,23-06,23-07,23-08,...,24-05,24-06,24-07,24-08,24-09,24-10,24-11,24-12,cluster_2023,cluster_2024
0,Алтайский край,Алейский муниципальный район,NaN,NaN,NaN,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
1,Алтайский край,Алтайский муниципальный район,1.0,1.0,1.0,4.0,1.0,0.0,0.0,0.0,...,1.0,4.0,4.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
2,Алтайский край,Баевский муниципальный район,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Алтайский край,Бийский муниципальный район,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
4,Алтайский край,Благовещенский муниципальный район,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0


In [12]:
df_2.head()

,region_name,mo,year,month,Средняя зарплата,population,urban_share,share_industry,share_agriculture,share_services,Здоровье_share,Маркетплейсы_share,Общественное питание_share,Продовольствие_share,Транспорт_share
0,Республика Саха (Якутия),Абыйский муниципальный район,2023,1,89695.4,3786.0,0.503170,0.172881,0.000000,0.782373,0.029033,0.027808,0.006422,0.457543,0.047340
1,Забайкальский край,Агинский муниципальный район,2023,1,39726.9,15742.0,0.309173,0.058462,0.000000,0.774359,0.056820,0.050721,0.037483,0.401458,0.123655
2,Республика Татарстан,Агрызский муниципальный район,2023,1,50675.0,34692.0,0.572466,0.140458,0.053680,0.786926,0.055180,0.057897,0.034938,0.426464,0.064718
3,Республика Татарстан,Азнакаевский муниципальный район,2023,1,54833.4,57623.0,0.727158,0.324417,0.066122,0.609382,0.052780,0.074096,0.045448,0.424763,0.070066
4,Республика Татарстан,Аксубаевский муниципальный район,2023,1,39374.8,26484.0,0.342773,0.127336,0.087909,0.739778,0.059539,0.086755,0.016397,0.484107,0.080304


In [13]:
month_cols = [
    c for c in df.columns
    if c.startswith(('23-', '24-'))
]

clusters_long = df.melt(
    id_vars=['region_name', 'mo'],
    value_vars=month_cols,
    var_name='month',
    value_name='cluster'
)

clusters_long = clusters_long.dropna(subset=['cluster']).copy()
clusters_long['cluster'] = clusters_long['cluster'].astype(int)

df_plot = df_2.copy()
df_plot['month'] = (
    (df_plot['year'] % 100).astype(str).str.zfill(2)
    + '-'
    + df_plot['month'].astype(str).str.zfill(2)
)

df_plot = clusters_long.merge(
    df_plot,
    on=['region_name', 'mo', 'month'],
    how='inner'
)

df_plot = df_plot.dropna(
    subset=[
        'Средняя зарплата',
        'Продовольствие_share'
    ]
)

cluster_colors = {
    0: '#E53935',
    1: '#FDD835',
    2: '#43A047',
    3: '#1E88E5',
    4: '#FB8C00',
    5: '#8E44AD'
}

In [17]:
import plotly.graph_objects as go
import pandas as pd


y_features = [
    'Продовольствие_share',
    'Маркетплейсы_share',
    'Общественное питание_share',
    'Здоровье_share',
    'Транспорт_share',
    'urban_share',
    'share_industry',
    'share_agriculture',
    'share_services',
    'population',
]

labels = {
    'Средняя зарплата': 'Средняя зарплата, руб.',
    'population': 'Население, чел.',
    'urban_share': 'Доля городского населения',
    'share_industry': 'Доля занятых в промышленности',
    'share_agriculture': 'Доля занятых в сельском хозяйстве',
    'share_services': 'Доля занятых в сфере услуг',
    'Здоровье_share': 'Доля расходов на здоровье',
    'Маркетплейсы_share': 'Доля расходов на маркетплейсы',
    'Общественное питание_share': 'Доля расходов на общественное питание',
    'Продовольствие_share': 'Доля расходов на продовольствие',
    'Транспорт_share': 'Доля расходов на транспорт',
}

x_feat = 'Средняя зарплата'
y0 = 'Продовольствие_share'

for col in ['Средняя зарплата', 'population']:
    if col in df_plot.columns:
        df_plot[col] = df_plot[col].round(0).astype('int32')

share_cols = [
    'Здоровье_share', 'Маркетплейсы_share', 'Общественное питание_share',
    'Продовольствие_share', 'Транспорт_share',
    'urban_share', 'share_industry', 'share_agriculture', 'share_services',
]
for col in share_cols:
    if col in df_plot.columns:
        df_plot[col] = df_plot[col].round(3).astype('float32')

clusters = sorted(df_plot['cluster'].unique())
month_cols = sorted(df_plot['month'].unique())

fig = go.Figure()

for month in month_cols:
    month_df = df_plot[df_plot['month'] == month]

    for cluster in clusters:
        d = month_df[month_df['cluster'] == cluster]

        fig.add_trace(
            go.Scatter(
                x=d[x_feat],
                y=d[y0],
                mode='markers',
                name=f'Кластер {cluster}',
                legendgroup=f'cluster-{cluster}',
                showlegend=False,
                marker=dict(
                    color=cluster_colors[cluster],
                    size=8,
                    opacity=0.75,
                ),
                customdata=d[['region_name', 'mo', 'month', 'cluster']].values,
                hovertemplate=(
                    '%{customdata[0]}<br>'
                    '%{customdata[1]}<br>'
                    'Месяц: %{customdata[2]}<br>'
                    'Кластер: %{customdata[3]}<br>'
                    'Зарплата: %{x:,.0f} руб.<br>'
                    'Значение: %{y:,.3f}'
                    '<extra></extra>'
                ),
                visible=(month == month_cols[0]),
            )
        )

n_clusters = len(clusters)
n_traces = len(fig.data)

for cluster in clusters:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            name=f'Кластер {cluster}',
            legendgroup=f'cluster-{cluster}',
            showlegend=True,
            marker=dict(
                color=cluster_colors[cluster],
                size=8,
                opacity=0.75,
            ),
            hoverinfo='skip',
            visible=True,
        )
    )

n_dummy = len(clusters)

steps = []
for i, month in enumerate(month_cols):
    visible = [False] * n_traces + [True] * n_dummy
    start = i * n_clusters
    end = start + n_clusters
    visible[start:end] = [True] * n_clusters

    steps.append(dict(
        method='update',
        args=[
            {'visible': visible},
            {'title.text': f'Зависимость переменных от средней зарплаты — {month}'},
        ],
        label=month,
    ))

y_cache = {}
for y_feat in y_features:
    y_cache[y_feat] = []
    for month in month_cols:
        month_df = df_plot[df_plot['month'] == month]
        for cluster in clusters:
            d = month_df[month_df['cluster'] == cluster]
            y_cache[y_feat].append(d[y_feat].tolist())
    y_cache[y_feat].extend([[None]] * n_dummy)

y_buttons = []
for y_feat in y_features:
    y_buttons.append(dict(
        label=labels[y_feat],
        method='update',
        args=[
            {'y': y_cache[y_feat]},
            {'yaxis.title.text': labels[y_feat]},
        ],
    ))

fig.update_layout(
    title=dict(
        text=f'Зависимость переменных от средней зарплаты — {month_cols[0]}',
        x=0.5,
        xanchor='center',
        y=0.97,
    ),
    template='plotly_white',
    autosize=True,

    margin=dict(t=140, l=60, r=30, b=100),

    xaxis=dict(title=labels[x_feat], tickformat=','),
    yaxis=dict(title=labels[y0]),

    legend=dict(title='Кластер'),

    updatemenus=[
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0.0, xanchor='left',
            y=1.14, yanchor='top',
            bgcolor='#dff5b5',
            bordercolor='#2f5d3a',
            font=dict(color='#2f5d3a', size=12),
            pad={'r': 10, 't': 5},
        ),
    ],

    sliders=[
        dict(
            active=0,
            currentvalue={
                'prefix': 'Месяц: ',
                'font': {'color': '#2f5d3a', 'size': 14},
            },
            bgcolor='#dff5b5',
            bordercolor='#2f5d3a',
            borderwidth=1,
            tickcolor='#2f5d3a',
            font=dict(color='#2f5d3a', size=11),
            activebgcolor='#a3e635',
            pad={'t': 40},
            steps=steps,
        )
    ],
)


fig.write_html(
    'wage_scatter_cluster.html',
    include_plotlyjs='cdn',
    full_html=True,
    config={
        'responsive': True,
    },
)

In [15]:
import plotly.graph_objects as go


cluster_share = (
    df_plot
    .dropna(subset=['cluster'])
    .groupby(['month', 'cluster'])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='share')
)

month_cols = sorted(
    df_plot['month'].dropna().unique(),
    key=lambda x: (
        int(str(x).split('-')[0]),
        int(str(x).split('-')[1])
    )
)


cluster_colors = {
    0: '#E53935',
    1: '#FDD835',
    2: '#43A047',
    3: '#1E88E5',
    4: '#FB8C00'
}

fig = go.Figure()


for cluster in range(5):
    d = cluster_share[
        cluster_share['cluster'] == cluster
    ]

    d = (
        d.set_index('month')
         .reindex(month_cols)
         .reset_index()
    )

    fig.add_trace(go.Scatter(
        x=d['month'],
        y=d['share'],
        mode='lines',
        name=f'Кластер {cluster}',
        stackgroup='one',
        line=dict(color=cluster_colors[cluster])
    ))

fig.update_layout(
    title='Изменение структуры кластеров',
    xaxis_title='Месяц',
    yaxis_title='Доля муниципалитетов, %',
    yaxis=dict(range=[0, 100]),
    template='plotly_white',
    height=550,
    hovermode='x unified'
)

fig.write_html(
    'cluster_profiles_change_2023.html',
    include_plotlyjs='cdn',
    full_html=True,
    config={
        'responsive': True,
    },
)
fig.show()

In [21]:
x_feat = 'Средняя зарплата'
y0 = 'Продовольствие_share'
y_features = [
    'Продовольствие_share',
    'Маркетплейсы_share',
    'Общественное питание_share',
    'Здоровье_share',
    'Транспорт_share',
    'urban_share',
    'share_industry',
    'share_agriculture',
    'share_services',
    'population'
]
labels = {
    'Средняя зарплата': 'Средняя зарплата, руб.',
    'population': 'Население, чел.',
    'urban_share': 'Доля городского населения',
    'share_industry': 'Доля занятых в промышленности',
    'share_agriculture': 'Доля занятых в сельском хозяйстве',
    'share_services': 'Доля занятых в сфере услуг',
    'Здоровье_share': 'Доля расходов на здоровье',
    'Маркетплейсы_share': 'Доля расходов на маркетплейсы',
    'Общественное питание_share': 'Доля расходов на общественное питание',
    'Продовольствие_share': 'Доля расходов на продовольствие',
    'Транспорт_share': 'Доля расходов на транспорт'
}

df_plot = df_plot.copy()
clusters = sorted(df_plot['cluster'].dropna().unique())
month_cols = sorted(df_plot['month'].dropna().unique())

def get_cluster_data(month, cluster, y_feat):
    d = df_plot[
        (df_plot['month'] == month) &
        (df_plot['cluster'] == cluster)
    ].copy()
    return d

fig = go.Figure()
for cluster in clusters:
    d = get_cluster_data(
        month_cols[0],
        cluster,
        y0
    )
    fig.add_trace(
        go.Scatter(
            x=d[x_feat],
            y=d[y0],
            mode='markers',
            name=f'Кластер {cluster}',
            legendgroup=f'cluster-{cluster}',
            showlegend=True,

            marker=dict(
                color=cluster_colors[cluster],
                size=8,
                opacity=0.75
            ),
            customdata=d[
                ['region_name', 'mo', 'month', 'cluster']
            ].values,
            hovertemplate=(
                '%{customdata[0]}<br>'
                '%{customdata[1]}<br>'
                'Месяц: %{customdata[2]}<br>'
                'Кластер: %{customdata[3]}<br>'
                'Зарплата: %{x:,.0f} руб.<br>'
                'Значение: %{y:,.3f}'
                '<extra></extra>'
            )
        )
    )

for cluster in clusters:
    d = get_cluster_data(
        month_cols[0],
        cluster,
        y0
    )

    if len(d) == 0:
        x_mean = []
        y_mean = []
        customdata = []
    else:
        x_mean = [d[x_feat].mean()]
        y_mean = [d[y0].mean()]
        customdata = [[cluster, len(d)]]
    fig.add_trace(
        go.Scatter(
            x=x_mean,
            y=y_mean,
            mode='markers',
            name=f'Среднее кластера {cluster}',
            legendgroup=f'cluster-mean-{cluster}',
            showlegend=False,

            marker=dict(
                color=cluster_colors[cluster],
                size=20,
                symbol='diamond',
                opacity=1,
                line=dict(
                    color='#1a1a1a',
                    width=2
                )
            ),
            customdata=customdata,
            hovertemplate=(
                '<b>Среднее кластера %{customdata[0]}</b><br>'
                'Количество МО: %{customdata[1]}<br>'
                'Средняя зарплата: %{x:,.0f} руб.<br>'
                'Среднее значение: %{y:,.3f}'
                '<extra></extra>'
            )
        )
    )

n_clusters = len(clusters)
steps = []
for month in month_cols:
    args = []
    for cluster in clusters:
        d = get_cluster_data(
            month,
            cluster,
            y0
        )
        args.append(
            dict(
                x=[d[x_feat].tolist()],
                y=[d[y0].tolist()],
                customdata=[
                    d[
                        ['region_name', 'mo', 'month', 'cluster']
                    ].values
                ]
            )
        )
    for cluster in clusters:

        d = get_cluster_data(
            month,
            cluster,
            y0
        )
        if len(d) == 0:

            args.append(
                dict(
                    x=[[]],
                    y=[[]],
                    customdata=[[]]
                )
            )

        else:

            args.append(
                dict(
                    x=[[d[x_feat].mean()]],
                    y=[[d[y0].mean()]],
                    customdata=[
                        [[cluster, len(d)]]
                    ]
                )
            )
    steps.append(
        dict(
            method='update',

            args=[
                {
                    'x': [
                        item['x'][0]
                        for item in args
                    ],
                    'y': [
                        item['y'][0]
                        for item in args
                    ],
                    'customdata': [
                        item['customdata'][0]
                        for item in args
                    ]
                },
                {
                    'title.text':
                        f'Зависимость переменных от средней зарплаты — {month}'
                }
            ],
            label=month
        )
    )

y_buttons = []

for y_feat in y_features:

    x_values = []
    y_values = []
    customdata_values = []


    for cluster in clusters:
        d = get_cluster_data(
            month_cols[0],
            cluster,
            y_feat
        )
        x_values.append(
            d[x_feat if x_feat != 'Средняя зарплата' else x_feat].tolist()
        )
        y_values.append(
            d[y_feat].tolist()
        )
        customdata_values.append(
            d[
                ['region_name', 'mo', 'month', 'cluster']
            ].values
        )

    for cluster in clusters:
        d = get_cluster_data(
            month_cols[0],
            cluster,
            y_feat
        )
        if len(d) == 0:
            x_values.append([])
            y_values.append([])
            customdata_values.append([])

        else:

            x_values.append(
                [d[x_feat].mean()]
            )
            y_values.append(
                [d[y_feat].mean()]
            )
            customdata_values.append(
                [[cluster, len(d)]]
            )
    y_buttons.append(
        dict(
            label=labels[y_feat],
            method='update',
            args=[
                {
                    'x': x_values,
                    'y': y_values,
                    'customdata': customdata_values
                },
                {
                    'yaxis.title.text': labels[y_feat]
                }
            ]
        )
    )


fig.update_layout(
    title=dict(
        text=(
            f'Зависимость переменных от средней зарплаты — '
            f'{month_cols[0]}'
        ),
        x=0.5,
        xanchor='center',
        y=0.97
    ),

    template='plotly_white',
    autosize=True,
    margin=dict(
        t=140,
        l=60,
        r=30,
        b=100
    ),

    xaxis=dict(
        title=labels[x_feat],
        tickformat=','
    ),
    yaxis=dict(
        title=labels[y0]
    ),
    legend=dict(
        title='Кластер'
    ),
    updatemenus=[
        dict(
            buttons=y_buttons,
            direction='down',
            showactive=True,
            x=0,
            xanchor='left',
            y=1.14,
            yanchor='top',
            bgcolor='#dff5b5',
            bordercolor='#2f5d3a',
            font=dict(
                color='#2f5d3a',
                size=12
            ),

            pad=dict(
                r=10,
                t=5
            )
        )
    ],

    sliders=[
        dict(
            active=0,
            currentvalue=dict(
                prefix='Месяц: ',
                font=dict(
                    color='#2f5d3a',
                    size=14
                )
            ),
            bgcolor='#dff5b5',
            bordercolor='#2f5d3a',
            borderwidth=1,
            tickcolor='#2f5d3a',
            font=dict(
                color='#2f5d3a',
                size=11
            ),
            activebgcolor='#a3e635',
            pad=dict(
                t=40
            ),
            steps=steps
        )
    ]
)


fig.write_html(
    'wage_scatter_cluster.html',
    include_plotlyjs='cdn',
    full_html=True,
    config={
        'responsive': True
    }
)

fig.show()